# RAG with Query Rewriting using Ollama

**Thesis: Mitigasi Halusinasi pada RAG-LLM**

This notebook implements:
- RAG (Retrieval-Augmented Generation) using PubMedQA dataset
- Query Rewriting technique to improve retrieval quality
- Ollama for LLM (llama3.2) and embeddings (nomic-embed-text)

---

## 1. Setup & Dependencies

In [3]:
# Install required packages
# Run this in your terminal/command prompt if pip install fails in notebook:
#   pip install datasets faiss-cpu ollama numpy

import subprocess
import sys

def install_packages():
    packages = ["datasets", "faiss-cpu", "ollama", "numpy"]
    for package in packages:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])
    print("All packages installed!")

# Uncomment the line below to install packages:
# install_packages()

# Or run this in your terminal:
print("Run this command in your terminal if packages are not installed:")
print(f'  "{sys.executable}" -m pip install datasets faiss-cpu ollama numpy')
print(f"\nPython being used: {sys.executable}")
print(f"Python version: {sys.version}")

Run this command in your terminal if packages are not installed:
  "C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\python.exe" -m pip install datasets faiss-cpu ollama numpy

Python being used: C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\python.exe
Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [4]:
import os
import json
import numpy as np
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass

import faiss
import ollama
from datasets import load_dataset

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Configuration

In [5]:
# Ollama Models
LLM_MODEL = "llama3.2"           # For generation and query rewriting
EMBED_MODEL = "nomic-embed-text" # For embeddings

# RAG Parameters
TOP_K_RETRIEVAL = 5   # Number of documents to retrieve
CHUNK_SIZE = 512      # Max characters per chunk

# Dataset
DATASET_NAME = "qiaojin/PubMedQA"
DATASET_SUBSET = "pqa_labeled"  # Labeled subset with ground truth

print(f"LLM Model: {LLM_MODEL}")
print(f"Embedding Model: {EMBED_MODEL}")
print(f"Dataset: {DATASET_NAME} ({DATASET_SUBSET})")

LLM Model: llama3.2
Embedding Model: nomic-embed-text
Dataset: qiaojin/PubMedQA (pqa_labeled)


## 3. Verify Ollama Connection

In [6]:
def check_ollama_models():
    """Check if required Ollama models are available."""
    try:
        response = ollama.list()
        
        # Handle different ollama library versions
        models_list = response.get('models', []) if isinstance(response, dict) else response.models
        
        available = []
        for m in models_list:
            # Handle both dict and object formats
            if isinstance(m, dict):
                name = m.get('name', '').split(':')[0]
            else:
                name = getattr(m, 'model', getattr(m, 'name', '')).split(':')[0]
            if name:
                available.append(name)
        
        print(f"Available Ollama models: {available}")
        
        missing = []
        for model in [LLM_MODEL, EMBED_MODEL]:
            # Check if model name is in available (partial match)
            found = any(model in name or name in model for name in available)
            if not found:
                missing.append(model)
        
        if missing:
            print(f"\nMissing models: {missing}")
            print("Please pull them using these commands in terminal:")
            for m in missing:
                print(f"  ollama pull {m}")
            return False
        
        print("\nAll required models are available!")
        return True
        
    except Exception as e:
        print(f"Error connecting to Ollama: {e}")
        print("\nTroubleshooting:")
        print("1. Make sure Ollama is installed: https://ollama.com/download")
        print("2. Start Ollama: Run 'ollama serve' in terminal")
        print("3. Pull required models:")
        print(f"   ollama pull {LLM_MODEL}")
        print(f"   ollama pull {EMBED_MODEL}")
        return False

check_ollama_models()

Available Ollama models: ['nomic-embed-text', 'llama3.2']

All required models are available!


True

## 4. Load PubMedQA Dataset

In [7]:
def load_pubmedqa(subset: str = DATASET_SUBSET, max_samples: int = 500):
    """
    Load PubMedQA dataset.
    
    PubMedQA structure:
    - pubid: PubMed article ID
    - question: The question
    - context: Dict with 'contexts' (list of context strings) and 'labels' (OBJECTIVE, METHODS, etc.)
    - long_answer: Detailed answer
    - final_decision: yes/no/maybe
    """
    print(f"Loading PubMedQA dataset ({subset})...")
    
    dataset = load_dataset(DATASET_NAME, subset, trust_remote_code=True)
    
    # Get train split (labeled subset only has train)
    data = dataset['train']
    
    # Limit samples for faster processing
    if max_samples and len(data) > max_samples:
        data = data.select(range(max_samples))
    
    print(f"Loaded {len(data)} samples")
    print(f"\nSample keys: {data[0].keys()}")
    
    return data

# Load dataset
pubmedqa_data = load_pubmedqa(max_samples=500)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Loading PubMedQA dataset (pqa_labeled)...


Loaded 500 samples

Sample keys: dict_keys(['pubid', 'question', 'context', 'long_answer', 'final_decision'])


In [8]:
# Examine a sample
sample = pubmedqa_data[0]
print("=" * 60)
print("SAMPLE DATA")
print("=" * 60)
print(f"\nPubID: {sample['pubid']}")
print(f"\nQuestion: {sample['question']}")
print(f"\nContexts ({len(sample['context']['contexts'])} sections):")
for i, (ctx, label) in enumerate(zip(sample['context']['contexts'], sample['context']['labels'])):
    print(f"  [{label}]: {ctx[:150]}...")
print(f"\nFinal Decision: {sample['final_decision']}")
print(f"\nLong Answer: {sample['long_answer'][:300]}...")

SAMPLE DATA

PubID: 21645374

Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

Contexts (2 sections):
  [BACKGROUND]: Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in ...
  [RESULTS]: The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole...

Final Decision: yes

Long Answer: Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best of our knowledge, this is the first report of mitochondria and chloroplasts moving on transvacuolar str...


## 5. Prepare Document Chunks for RAG

In [9]:
@dataclass
class Document:
    """A document chunk for RAG."""
    text: str
    pubid: str
    question: str
    section_label: str
    answer: str
    decision: str

def prepare_documents(data) -> List[Document]:
    """
    Convert PubMedQA data into document chunks.
    Each context section becomes a separate document.
    """
    documents = []
    
    for item in data:
        pubid = str(item['pubid'])
        question = item['question']
        answer = item['long_answer']
        decision = item['final_decision']
        
        contexts = item['context']['contexts']
        labels = item['context']['labels']
        
        for ctx, label in zip(contexts, labels):
            doc = Document(
                text=ctx.strip(),
                pubid=pubid,
                question=question,
                section_label=label,
                answer=answer,
                decision=decision
            )
            documents.append(doc)
    
    print(f"Created {len(documents)} document chunks from {len(data)} samples")
    return documents

# Prepare documents
documents = prepare_documents(pubmedqa_data)

Created 1706 document chunks from 500 samples


## 6. Ollama Embedding Function

In [12]:
def get_embedding(text: str, model: str = EMBED_MODEL) -> np.ndarray:
    """
    Get embedding for a single text using Ollama.
    """
    response = ollama.embeddings(model=model, prompt=text)
    return np.array(response['embedding'], dtype='float32')

def get_embeddings_batch(texts: List[str], model: str = EMBED_MODEL, 
                         batch_size: int = 32, show_progress: bool = True) -> np.ndarray:
    """
    Get embeddings for multiple texts.
    """
    embeddings = []
    total = len(texts)
    
    for i in range(0, total, batch_size):
        batch = texts[i:i+batch_size]
        
        for text in batch:
            emb = get_embedding(text, model)
            embeddings.append(emb)
        
        if show_progress:
            progress = min(i + batch_size, total)
            print(f"\rEmbedding progress: {progress}/{total}", end="")
    
    if show_progress:
        print("\nDone!")
    
    return np.array(embeddings, dtype='float32')

# Test embedding
test_emb = get_embedding("This is a test sentence.")
print(f"Embedding dimension: {len(test_emb)}")

Embedding dimension: 768


## 7. Build FAISS Vector Index

In [13]:
def build_index(documents: List[Document]) -> Tuple[faiss.IndexFlatIP, np.ndarray]:
    """
    Build FAISS index from documents.
    Uses Inner Product (cosine similarity with normalized vectors).
    """
    print("Building vector index...")
    
    # Get all document texts
    texts = [doc.text for doc in documents]
    
    # Get embeddings
    embeddings = get_embeddings_batch(texts)
    
    # Normalize for cosine similarity
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    embeddings_normalized = embeddings / norms
    
    # Build FAISS index
    dim = embeddings_normalized.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings_normalized)
    
    print(f"Index built: {index.ntotal} vectors, dimension {dim}")
    
    return index, embeddings_normalized

# Build the index (this may take a few minutes)
index, doc_embeddings = build_index(documents)

Building vector index...
Embedding progress: 1706/1706
Done!
Index built: 1706 vectors, dimension 768


## 8. Query Rewriting with LLM

Query Rewriting reformulates user queries to improve retrieval quality. This helps when:
- The original query is ambiguous
- The query uses different terminology than the documents
- The query is too short or lacks context

In [14]:
QUERY_REWRITE_PROMPT = """You are a query rewriting assistant for a medical question-answering system.

Your task is to rewrite the user's question to improve retrieval from a medical research database.
The rewritten query should:
1. Be more specific and detailed
2. Include relevant medical terminology
3. Expand abbreviations if present
4. Maintain the original intent

Original question: {query}

Rewrite the question to be more effective for retrieving relevant medical research.
Output ONLY the rewritten question, nothing else.

Rewritten question:"""

def rewrite_query(query: str, model: str = LLM_MODEL) -> str:
    """
    Rewrite a query using LLM to improve retrieval quality.
    """
    prompt = QUERY_REWRITE_PROMPT.format(query=query)
    
    response = ollama.generate(
        model=model,
        prompt=prompt,
        options={
            'temperature': 0.3,  # Low temperature for consistent rewrites
            'num_predict': 150,
        }
    )
    
    rewritten = response['response'].strip()
    
    # Clean up any extra formatting
    rewritten = rewritten.replace('\n', ' ').strip()
    
    # If rewrite is empty or too short, return original
    if len(rewritten) < 10:
        return query
    
    return rewritten

# Test query rewriting
test_queries = [
    "Does vitamin D help with COVID?",
    "Is aspirin good for heart?",
    "Can exercise prevent diabetes?"
]

print("Query Rewriting Examples:")
print("=" * 60)
for q in test_queries:
    rewritten = rewrite_query(q)
    print(f"\nOriginal:  {q}")
    print(f"Rewritten: {rewritten}")

Query Rewriting Examples:

Original:  Does vitamin D help with COVID?
Rewritten: Does vitamin D supplementation have a beneficial effect on the severity and duration of COVID-19 in adults, particularly in those with vitamin D deficiency or insufficiency, as measured by clinical outcomes and serum 25-hydroxyvitamin D levels?

Original:  Is aspirin good for heart?
Rewritten: Is aspirin effective in reducing cardiovascular risk and preventing myocardial infarction in patients with coronary artery disease?

Original:  Can exercise prevent diabetes?
Rewritten: Can regular aerobic exercise, as defined by moderate-intensity cardio activity lasting at least 150 minutes per week, prevent the onset of type 2 diabetes mellitus in individuals with impaired glucose tolerance or prediabetes?


## 9. Retrieval Function

In [15]:
@dataclass
class RetrievalResult:
    """Result of document retrieval."""
    document: Document
    score: float

def retrieve(query: str, k: int = TOP_K_RETRIEVAL, 
             use_rewriting: bool = True) -> Tuple[List[RetrievalResult], str, str]:
    """
    Retrieve relevant documents for a query.
    
    Returns:
        - List of RetrievalResult
        - Original query
        - Rewritten query (same as original if rewriting disabled)
    """
    original_query = query
    
    # Query Rewriting
    if use_rewriting:
        rewritten_query = rewrite_query(query)
    else:
        rewritten_query = query
    
    # Get query embedding
    query_emb = get_embedding(rewritten_query)
    query_emb = query_emb / np.linalg.norm(query_emb)  # Normalize
    query_emb = query_emb.reshape(1, -1)
    
    # Search
    distances, indices = index.search(query_emb, k)
    
    # Build results
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx >= 0:  # Valid index
            results.append(RetrievalResult(
                document=documents[idx],
                score=float(dist)
            ))
    
    return results, original_query, rewritten_query

# Test retrieval
test_query = "Does exercise help prevent type 2 diabetes?"
results, orig, rewritten = retrieve(test_query)

print(f"Original Query: {orig}")
print(f"Rewritten Query: {rewritten}")
print(f"\nTop {len(results)} Retrieved Documents:")
print("=" * 60)
for i, r in enumerate(results, 1):
    print(f"\n[{i}] Score: {r.score:.4f}")
    print(f"    Section: {r.document.section_label}")
    print(f"    Text: {r.document.text[:200]}...")

Original Query: Does exercise help prevent type 2 diabetes?
Rewritten Query: Does regular aerobic exercise, such as brisk walking or cycling, have a significant impact on preventing the development of type 2 diabetes mellitus in adults with impaired glucose tolerance or insulin resistance?

Top 5 Retrieved Documents:

[1] Score: 0.6924
    Section: METHODS
    Text: In 60 middle-aged individuals without diabetes we studied the associations of fasting plasma glucose, 2-hour post oral glucose tolerance test plasma glucose, insulin sensitivity as well as body fat pe...

[2] Score: 0.6893
    Section: BACKGROUND
    Text: Several studies have shown associations between hyperglycemia and risk of cardiovascular disease (CVD) and mortality, yet glucose-lowering treatment does little to mitigate this risk. We examined whet...

[3] Score: 0.6819
    Section: BACKGROUND
    Text: Since insulin therapy might have an atherogenic effect, we studied the relationship between cumulative insulin dose a

## 10. Answer Generation with RAG

In [16]:
RAG_SYSTEM_PROMPT = """You are a medical question-answering assistant. 
Answer questions based ONLY on the provided context from medical research.
Be concise, accurate, and cite specific findings from the context.
If the context doesn't contain relevant information, say "I cannot find relevant information in the provided context."
Do not make up information or use knowledge outside the context."""

RAG_USER_PROMPT = """Context from medical research:
{context}

Question: {question}

Based on the context above, provide a concise and accurate answer:"""

def generate_answer(query: str, retrieved_docs: List[RetrievalResult], 
                    model: str = LLM_MODEL) -> str:
    """
    Generate answer using retrieved context.
    """
    # Build context from retrieved documents
    context_parts = []
    for i, r in enumerate(retrieved_docs, 1):
        context_parts.append(f"[{i}] ({r.document.section_label}): {r.document.text}")
    
    context = "\n\n".join(context_parts)
    
    # Generate answer
    response = ollama.chat(
        model=model,
        messages=[
            {'role': 'system', 'content': RAG_SYSTEM_PROMPT},
            {'role': 'user', 'content': RAG_USER_PROMPT.format(context=context, question=query)}
        ],
        options={
            'temperature': 0.1,  # Low temperature for factual responses
            'num_predict': 300,
        }
    )
    
    return response['message']['content'].strip()

## 11. Complete RAG Pipeline

In [17]:
@dataclass
class RAGResponse:
    """Complete RAG response with all components."""
    original_query: str
    rewritten_query: str
    answer: str
    retrieved_contexts: List[RetrievalResult]
    query_rewriting_enabled: bool

def rag_pipeline(query: str, use_rewriting: bool = True, 
                 top_k: int = TOP_K_RETRIEVAL) -> RAGResponse:
    """
    Complete RAG pipeline with optional query rewriting.
    
    Args:
        query: User's question
        use_rewriting: Whether to apply query rewriting
        top_k: Number of documents to retrieve
    
    Returns:
        RAGResponse with answer and metadata
    """
    # Step 1: Retrieve (with optional query rewriting)
    retrieved, original, rewritten = retrieve(query, k=top_k, use_rewriting=use_rewriting)
    
    # Step 2: Generate answer
    answer = generate_answer(query, retrieved)
    
    return RAGResponse(
        original_query=original,
        rewritten_query=rewritten,
        answer=answer,
        retrieved_contexts=retrieved,
        query_rewriting_enabled=use_rewriting
    )

def print_response(response: RAGResponse, show_contexts: bool = True):
    """
    Pretty print a RAG response.
    """
    print("\n" + "=" * 70)
    print("RAG RESPONSE")
    print("=" * 70)
    
    print(f"\nOriginal Query: {response.original_query}")
    
    if response.query_rewriting_enabled:
        print(f"Rewritten Query: {response.rewritten_query}")
    else:
        print("Query Rewriting: DISABLED")
    
    print(f"\n{'─' * 70}")
    print("ANSWER:")
    print(f"{'─' * 70}")
    print(response.answer)
    
    if show_contexts:
        print(f"\n{'─' * 70}")
        print(f"RETRIEVED CONTEXTS ({len(response.retrieved_contexts)} documents):")
        print(f"{'─' * 70}")
        for i, ctx in enumerate(response.retrieved_contexts, 1):
            print(f"\n[{i}] Score: {ctx.score:.4f} | Section: {ctx.document.section_label}")
            print(f"    {ctx.document.text[:300]}...")
    
    print("\n" + "=" * 70)

## 12. Test: Compare With vs Without Query Rewriting

In [18]:
# Test query
test_query = "Does vitamin D supplementation help with respiratory infections?"

print("\n" + "#" * 70)
print("# COMPARISON: WITH vs WITHOUT QUERY REWRITING")
print("#" * 70)

# Without query rewriting
print("\n>>> WITHOUT Query Rewriting:")
response_no_rewrite = rag_pipeline(test_query, use_rewriting=False)
print_response(response_no_rewrite, show_contexts=False)

# With query rewriting
print("\n>>> WITH Query Rewriting:")
response_with_rewrite = rag_pipeline(test_query, use_rewriting=True)
print_response(response_with_rewrite, show_contexts=False)


######################################################################
# COMPARISON: WITH vs WITHOUT QUERY REWRITING
######################################################################

>>> WITHOUT Query Rewriting:

RAG RESPONSE

Original Query: Does vitamin D supplementation help with respiratory infections?
Query Rewriting: DISABLED

──────────────────────────────────────────────────────────────────────
ANSWER:
──────────────────────────────────────────────────────────────────────
I cannot find any information in the provided context that suggests vitamin D supplementation helps with respiratory infections. The context only discusses the relationship between vitamin D and autoimmune diseases, and the vitamin D status of prepubertal children with CD (Crohn's disease), but does not mention its effect on respiratory infections.


>>> WITH Query Rewriting:

RAG RESPONSE

Original Query: Does vitamin D supplementation help with respiratory infections?
Rewritten Query: Does vitamin D s

## 13. Interactive Chat Interface

In [19]:
def start_chat(use_rewriting: bool = True, show_contexts: bool = False):
    """
    Start interactive chat with the RAG system.
    
    Args:
        use_rewriting: Enable query rewriting (default: True)
        show_contexts: Show retrieved contexts in output (default: False)
    """
    print("\n" + "=" * 70)
    print("RAG CHATBOT - PubMedQA (with Query Rewriting)")
    print("=" * 70)
    print(f"\nSettings:")
    print(f"  - Query Rewriting: {'ENABLED' if use_rewriting else 'DISABLED'}")
    print(f"  - Show Contexts: {'YES' if show_contexts else 'NO'}")
    print(f"  - LLM Model: {LLM_MODEL}")
    print(f"  - Embedding Model: {EMBED_MODEL}")
    print(f"\nCommands:")
    print("  - Type 'exit' or 'quit' to end the session")
    print("  - Type 'toggle rewrite' to enable/disable query rewriting")
    print("  - Type 'toggle context' to show/hide retrieved contexts")
    print("=" * 70)
    
    while True:
        try:
            user_input = input("\n\nYou: ").strip()
        except EOFError:
            print("\nSession ended.")
            break
        
        if not user_input:
            continue
        
        # Handle commands
        if user_input.lower() in ['exit', 'quit', 'q']:
            print("\nGoodbye!")
            break
        
        if user_input.lower() == 'toggle rewrite':
            use_rewriting = not use_rewriting
            print(f"\nQuery Rewriting: {'ENABLED' if use_rewriting else 'DISABLED'}")
            continue
        
        if user_input.lower() == 'toggle context':
            show_contexts = not show_contexts
            print(f"\nShow Contexts: {'YES' if show_contexts else 'NO'}")
            continue
        
        # Process query
        try:
            response = rag_pipeline(user_input, use_rewriting=use_rewriting)
            
            print("\n" + "-" * 50)
            
            if use_rewriting and response.rewritten_query != response.original_query:
                print(f"Rewritten Query: {response.rewritten_query}")
                print("-" * 50)
            
            print(f"\nBot: {response.answer}")
            
            if show_contexts:
                print(f"\n[Retrieved {len(response.retrieved_contexts)} contexts]")
                for i, ctx in enumerate(response.retrieved_contexts[:3], 1):
                    print(f"  [{i}] {ctx.document.text[:150]}...")
                    
        except Exception as e:
            print(f"\nError: {e}")
            print("Please try again with a different query.")

# Uncomment to start interactive chat:
# start_chat(use_rewriting=True, show_contexts=False)

In [20]:
# Start the interactive chat
# You can change the settings:
#   use_rewriting=True  -> Enable query rewriting
#   show_contexts=True  -> Show retrieved document chunks

start_chat(use_rewriting=True, show_contexts=True)


RAG CHATBOT - PubMedQA (with Query Rewriting)

Settings:
  - Query Rewriting: ENABLED
  - Show Contexts: YES
  - LLM Model: llama3.2
  - Embedding Model: nomic-embed-text

Commands:
  - Type 'exit' or 'quit' to end the session
  - Type 'toggle rewrite' to enable/disable query rewriting
  - Type 'toggle context' to show/hide retrieved contexts




You:  Does aspirin reduce risk of stroke?



--------------------------------------------------
Rewritten Query: Does low-dose acetylsalicylic acid (aspirin) therapy reduce the risk of ischemic stroke in patients with a history of cardiovascular disease or diabetes mellitus?
--------------------------------------------------

Bot: I cannot find relevant information in the provided context about aspirin reducing the risk of stroke. The context discusses the effects of statins on stroke risk, the relationship between insulin therapy and atherosclerosis, and the relationship between hyperglycemia and cardiovascular disease, but does not mention aspirin.

[Retrieved 5 contexts]
  [1] In primary and secondary prevention trials, statins have been shown to reduce the risk of stroke. In addition to lipid lowering, statins have a number...
  [2] Since insulin therapy might have an atherogenic effect, we studied the relationship between cumulative insulin dose and atherosclerosis in type 1 diab...
  [3] We conducted a population-based cas



You:  Does caffeine intake affect sleep quality?



--------------------------------------------------
Rewritten Query: Does moderate caffeine consumption (defined as 200-400 mg per day) impact sleep quality in adults, particularly in relation to sleep latency, sleep duration, and sleep architecture, in individuals with and without pre-existing sleep disorders?
--------------------------------------------------

Bot: I cannot find relevant information in the provided context about the effect of caffeine intake on sleep quality.

[Retrieved 5 contexts]
  [1] Anchoring vignettes are brief texts describing a hypothetical character who illustrates a certain fixed level of a trait under evaluation. This resear...
  [2] The prevalence of self-reported problems with sleep and energy was 53 %. Without correction of cut-point shifts, age, sex, and the number of comorbidi...
  [3] To investigate the effect of fenofibrate on sleep apnoea indices....




You:  exit



Goodbye!


## 14. Evaluation Helper (for Thesis)

This section provides helper functions for evaluating the RAG system with and without query rewriting.

In [24]:
def evaluate_sample(sample_idx: int, use_rewriting: bool = True) -> Dict:
    """
    Evaluate RAG on a single PubMedQA sample.
    
    Returns dict with:
        - question
        - ground_truth_answer
        - ground_truth_decision
        - generated_answer
        - rewritten_query (if enabled)
        - retrieval_scores
    """
    sample = pubmedqa_data[sample_idx]
    question = sample['question']
    
    # Get RAG response
    response = rag_pipeline(question, use_rewriting=use_rewriting)
    
    return {
        'question': question,
        'ground_truth_answer': sample['long_answer'],
        'ground_truth_decision': sample['final_decision'],
        'generated_answer': response.answer,
        'rewritten_query': response.rewritten_query if use_rewriting else None,
        'retrieval_scores': [r.score for r in response.retrieved_contexts],
        'use_rewriting': use_rewriting
    }

# Example: Evaluate on first 3 samples
print("Evaluation Examples:")
print("=" * 60)

for i in range(3):
    result = evaluate_sample(i, use_rewriting=True)
    print(f"\n[Sample {i}]")
    print(f"Question: {result['question'][:100]}...")
    print(f"Rewritten: {result['rewritten_query'][:100] if result['rewritten_query'] else 'N/A'}...")
    print(f"Ground Truth Decision: {result['ground_truth_decision']}")
    print(f"Generated Answer: {result['generated_answer'][:150]}...")
    print(f"Avg Retrieval Score: {np.mean(result['retrieval_scores']):.4f}")

Evaluation Examples:

[Sample 0]
Question: Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?...
Rewritten: Do mitochondria play a role in regulating mitochondrial dynamics and morphological changes in plant ...
Ground Truth Decision: yes
Generated Answer: Based on the provided context, it is not explicitly stated that mitochondria play a direct role in remodelling lace plant leaves during programmed cel...
Avg Retrieval Score: 0.7215

[Sample 1]
Question: Landolt C and snellen e acuity: differences in strabismus amblyopia?...
Rewritten: What are the differences in Landolt C and Snellen E acuity thresholds between patients with strabism...
Ground Truth Decision: no
Generated Answer: According to the context, in the eyes with strabismus amblyopia, the mean difference between Landolt C (LR) and Snellen E (SE) acuity was 0.55 lines, ...
Avg Retrieval Score: 0.7739

[Sample 2]
Question: Syncope during bathing in infants, a pediatric form of water-ind

## 15. Save/Load Index (Optional)

Save the FAISS index to avoid re-embedding every time.

In [25]:
import pickle

def save_index(index, documents, filepath: str = "pubmedqa_index.pkl"):
    """
    Save FAISS index and documents to file.
    """
    data = {
        'index': faiss.serialize_index(index),
        'documents': documents
    }
    with open(filepath, 'wb') as f:
        pickle.dump(data, f)
    print(f"Index saved to {filepath}")

def load_index(filepath: str = "pubmedqa_index.pkl"):
    """
    Load FAISS index and documents from file.
    """
    with open(filepath, 'rb') as f:
        data = pickle.load(f)
    
    index = faiss.deserialize_index(data['index'])
    documents = data['documents']
    
    print(f"Loaded index with {index.ntotal} vectors")
    return index, documents

# Uncomment to save:
# save_index(index, documents, "pubmedqa_index.pkl")

# Uncomment to load:
# index, documents = load_index("pubmedqa_index.pkl")

In [26]:
# Uncomment to save:
save_index(index, documents, "pubmedqa_index.pkl")

Index saved to pubmedqa_index.pkl


---

## Summary

This notebook implements:

1. **RAG Pipeline** using PubMedQA dataset with Ollama
2. **Query Rewriting** - reformulates user queries before retrieval
3. **Vector Search** using FAISS with Ollama embeddings
4. **Interactive Chat** with toggle options

### Next Steps for Thesis:

1. **Add Context Reranking** - Use CrossEncoder to reorder retrieved documents
2. **Add Active Hallucination Detection** - Verify generated answers against context
3. **Implement RAGAS Evaluation** - Measure faithfulness, relevance, etc.
4. **Compare Configurations** - Test all combinations of techniques

---